## Train a character-level GPT on some text data

The inputs here are simple text files, which we chop up to individual characters and then train GPT on. So you could say this is a char-transformer instead of a char-rnn. Doesn't quite roll off the tongue as well. In this example we will feed it some Shakespeare, which we'll get it to predict character-level.

In [11]:
# set up logging
import logging
logging.basicConfig(
        format="%(asctime)s - %(levelname)s - %(name)s:  %(message)s",
        datefmt="%m/%d/%Y %H:%M:%S",
        level=logging.INFO,
)

In [12]:
# make deterministic
from mingpt.utils import set_seed
set_seed(42)

In [13]:
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F

In [14]:
%%writefile dataset.py

import math
import torch
from torch.utils.data import Dataset

class CharDataset(Dataset):

    def __init__(self, data, block_size):
        chars = sorted(list(set(data)))
        data_size, vocab_size = len(data), len(chars)
        print('data has %d characters, %d unique.' % (data_size, vocab_size))
        
        self.stoi = { ch:i for i,ch in enumerate(chars) }
        self.itos = { i:ch for i,ch in enumerate(chars) }
        self.block_size = block_size
        self.vocab_size = vocab_size
        self.data = data
    
    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        # grab a chunk of (block_size + 1) characters from the data
        chunk = self.data[idx:idx + self.block_size + 1]
        # encode every character to an integer
        dix = [self.stoi[s] for s in chunk]
        """
        arrange data and targets so that the first i elements of x
        will be asked to predict the i-th element of y. Notice that
        the eventual language model will actually make block_size
        individual predictions at the same time based on this data,
        so we are being clever and amortizing the cost of the forward
        pass of the network. So for example if block_size is 4, then
        we could e.g. sample a chunk of text "hello", the integers in
        x will correspond to "hell" and in y will be "ello". This will
        then actually "multitask" 4 separate examples at the same time
        in the language model:
        - given just "h", please predict "e" as next
        - given "he" please predict "l" next
        - given "hel" predict "l" next
        - given "hell" predict "o" next
        
        In addition, because the DataLoader will create batches of examples,
        every forward/backward pass during training will simultaneously train
        a LOT of predictions, amortizing a lot of computation. In particular,
        for a batched input of integers X (B, T) where B is batch size and
        T is block_size and Y (B, T), the network will during training be
        simultaneously training to make B*T predictions, all at once! Of course,
        at test time we can paralellize across batch B, but unlike during training
        we cannot parallelize across the time dimension T - we have to run
        a forward pass of the network to recover the next single character of the 
        sequence along each batch dimension, and repeatedly always feed in a next
        character to get the next one.
        
        So yes there is a big asymmetry between train/test time of autoregressive
        models. During training we can go B*T at a time with every forward pass,
        but during test time we can only go B at a time, T times, with T forward 
        passes.
        """
        x = torch.tensor(dix[:-1], dtype=torch.long)
        y = torch.tensor(dix[1:], dtype=torch.long)
        return x, y


Overwriting dataset.py


In [15]:
block_size = 128 # spatial extent of the model for its context

In [16]:
!curl https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -o input.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1089k  100 1089k    0     0  2423k      0 --:--:-- --:--:-- --:--:-- 2420k


In [17]:
# you can download this file at https://github.com/karpathy/char-rnn/blob/master/data/tinyshakespeare/input.txt
from dataset import CharDataset
text = open('input.txt', 'r').read() # don't worry we won't run out of file handles
train_dataset = CharDataset(text, block_size) # one line of poem is roughly 50 characters

data has 1115394 characters, 65 unique.


In [18]:
from mingpt.model import GPT, GPTConfig
mconf = GPTConfig(train_dataset.vocab_size, train_dataset.block_size,
                  n_layer=8, n_head=8, n_embd=512)
model = GPT(mconf)

08/06/2026 12:23:22 - INFO - mingpt.model:  number of parameters: 2.535219e+07


In [19]:
from mingpt.trainer import Trainer, TrainerConfig

# initialize a trainer instance and kick off training
tconf = TrainerConfig(max_epochs=2, batch_size=512, learning_rate=6e-4,
                      lr_decay=True, warmup_tokens=512*20, final_tokens=2*len(train_dataset)*block_size,
                      num_workers=4)
trainer = Trainer(model, train_dataset, None, tconf)
trainer.train()

  0%|          | 0/2179 [00:00<?, ?it/s]/home/lpy/workspace/misc/tutorial/CS224n-Natural-Language-Processing-2024-Spring-Assignment/.venv/lib/python3.12/site-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
epoch 1 iter 2178: train loss 0.30785. lr 3.000169e-04: 100%|██████████| 2179/2179 [10:33<00:00,  3.44it/s]
epoch 2 iter 2178: train loss 0.16813. lr 6.000000e-05: 100%|██████████| 2179/2179 [10:17<00:00,  3.53it/s]


In [23]:
# alright, let's sample some character-level Shakespeare
from mingpt.utils import sample

context = "Peiyu Liu "
x = torch.tensor([train_dataset.stoi[s] for s in context], dtype=torch.long)[None,...].to(trainer.device)
y = sample(model, x, 2000, temperature=1.0, sample=True, top_k=10)[0]
completion = ''.join([train_dataset.itos[int(i)] for i in y])
print(completion)

Peiyu Liu does not all Marcius for yourself,
To lose your way out of doors!

VOLUMNIA:
She shall, she shall.

VIRGILIA:
Indeed, no, by your patience; I'll not over the
threshold till my lord return from the wars.

VALERIA:
Fie, you confine yourself most unreasonably: come,
you must go visit the good lady that lies in.

VIRGILIA:
I will wish her speedy strength, and visit her with
my prayers; but I cannot go thither.

VOLUMNIA:
Why, I pray you?

VIRGILIA:
'Tis not to save labour, nor that I want love.

VALERIA:
You would be another Penelope: yet, they say, all
the yarn she spun in Ulysses' absence did but fill
Ithaca full of moths. Come; I would your cambric
were sensible as your finger, that you might leave
pricking it for pity. Come, you shall go with us.

VIRGILIA:
No, good madam, pardon me; indeed, I will not forth.

VALERIA:
In truth, la, go with me; and I'll tell you
excellent news of your husband.

VIRGILIA:
O, good madam, there can be none yet.

VALERIA:
Verily, I do not jest wi

In [11]:
# well that was fun